In [2]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "groq", "sentence-transformers", "faiss-cpu", "requests",
    "beautifulsoup4","pandas", "ipywidgets", "transformers", "sentencepiece", "sacremoses", "torch"], capture_output=True)

from groq import Groq
from sentence_transformers import SentenceTransformer
from transformers import MarianMTModel, MarianTokenizer
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import torch, faiss, numpy as np, time, requests, pandas as pd
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, clear_output

GROQ_API_KEY = "your_groq_api_key_here"
client  = Groq(api_key=GROQ_API_KEY)
MODEL_A = "llama-3.3-70b-versatile"
MODEL_B = "llama-3.1-8b-instant"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚙️ Device- الجهاز: {'GPU ✅' if torch.cuda.is_available() else 'CPU'}")

print("🔄 RAG model- Download - جاري تحميل مودل ...")
emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print("🔄 Translation model- Download -  جاري تحميل مودل ...")
tok1 = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-ar-en")
mod1 = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-ar-en").to(device)
tok2 = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-ar")
mod2 = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-ar").to(device)
tok3 = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
mod3 = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt").to(device)

print("✅ Ready - Done👏🏻  !")


⚙️ Device- الجهاز: GPU ✅
🔄 RAG model- Download - جاري تحميل مودل ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

✅ Ready - Done👏🏻  !


In [5]:
def get_text(src):
    if src.startswith("http"):
        r = requests.get(src, headers={"User-Agent":"Mozilla/5.0"}, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")
        for t in soup(["script","style","nav","footer"]): t.decompose()
        return " ".join(soup.get_text(" ").split())

    return src.strip()

def build_index(text):
    chunks = [text[i:i+500] for i in range(0, len(text), 400)]
    vecs = emb.encode(chunks, normalize_embeddings=True).astype("float32")
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return chunks, idx

def ask(model, question, context):
    prompt = f"أجب بنفس لغة السؤال. Answe in the same context.\n\nسياق:\n{context}\n\n Quastion - سؤال: {question}\n\nجواب:"
    t0 = time.time()
    out = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":prompt}],
        max_tokens=512, temperature=0.2
    )
    return out.choices[0].message.content, round(time.time()-t0,2), out.usage.completion_tokens

# Text-input window
source_box   = widgets.Textarea(placeholder="النص - الرابط  ", layout=widgets.Layout(width="100%", height="90px"))
question_box = widgets.Text(placeholder=" سؤالك ...", layout=widgets.Layout(width="100%"))
rag_btn      = widgets.Button(description="🔍  Search", button_style="primary", layout=widgets.Layout(width="100%", height="40px"))
rag_out      = widgets.Output()

def on_rag(b):
    with rag_out:
        clear_output()
        src = source_box.value.strip()
        q   = question_box.value.strip()
        if not src or not q:
            print("⚠️ Source of quastion!"); return
        print("Please wait - انتظر من فضلك...")
        text = get_text(src)
        chunks, idx = build_index(text)
        print(f"✅ {len(chunks)} Part -قطعه\n")
        qv = emb.encode([q], normalize_embeddings=True).astype("float32")
        _, I = idx.search(qv, 3)
        ctx = " ".join(chunks[i] for i in I[0])
        ans_a, ta, tok_a = ask(MODEL_A, q, ctx)
        ans_b, tb, tok_b = ask(MODEL_B, q, ctx)
        pd.set_option("display.max_colwidth", None)
        display(pd.DataFrame({
            "Model - النموذج":   ["🦙 LLaMA 3.3 70B", "⚡ LLaMA 3.1 8B"],
            "Answer - الجواب":    [ans_a, ans_b],
            "Time -الوقت (s)": [ta, tb],
            "Tokens - التوكنز":   [tok_a, tok_b]
        }).style.set_properties(subset=["Answer - الجواب"], **{ # Changed 'Answer-الجواب' to 'Answer - الجواب'
            "white-space":"pre-wrap","text-align":"right","width":"500px"
        }).set_table_styles([{"selector":"th","props":[("background","#1a1a2e"),("color","white"),("text-align","center")]}]))

rag_btn.on_click(on_rag)

tab1 = widgets.VBox([
    widgets.HTML("<h3 style='text-align:center'> 🔍Q & A ⁉️ </h3>"),
    widgets.Label("📌Source المصدر (URL / text ):"), source_box,
    widgets.Label("❓Quastion السؤال:"), question_box,
    rag_btn, rag_out
])

# Transaltion window
trans_input = widgets.Textarea(placeholder="  اكتب النص للترجمة... | Enter text to translate...", layout=widgets.Layout(width="100%", height="90px"))
trans_btn   = widgets.Button(description="🌐  Translate - ترجم", button_style="success", layout=widgets.Layout(width="100%", height="40px"))
trans_out   = widgets.Output()

def on_trans(b):
    with trans_out:
        clear_output()
        txt = trans_input.value.strip()
        if not txt:
            print("⚠️put your text - اكتب نصاً!"); return
        try:
            arabic_chars = sum(1 for c in txt if '\u0600' <= c <= '\u06ff')
            lang = "ar" if arabic_chars > len(txt) * 0.3 else "en"

            if lang == "ar":
                cur_tok, cur_mod = tok1, mod1
                s_lang, t_lang = "ar_AR", "en_XX"
                flag = "🇸🇦 عربي → إنجليزي 🇬🇧"
            else:
                cur_tok, cur_mod = tok2, mod2
                s_lang, t_lang = "en_XX", "ar_AR"
                flag = "🇬🇧 English → Arabic 🇸🇦"

            print(f"🔍 اللغة Language: {flag}\n")

            t0 = time.time()
            inp1 = cur_tok(txt, return_tensors="pt").to(device)
            res1 = cur_tok.decode(cur_mod.generate(**inp1)[0], skip_special_tokens=True)
            t1   = round(time.time()-t0, 3)

            tok3.src_lang = s_lang
            t0 = time.time()
            inp3 = tok3(txt, return_tensors="pt").to(device)
            res2 = tok3.batch_decode(mod3.generate(**inp3, forced_bos_token_id=tok3.lang_code_to_id[t_lang]), skip_special_tokens=True)[0]
            t2   = round(time.time()-t0, 3)

            faster =  "Helsinki" if t1 < t2 else "mBART"

            display(widgets.HTML(f"""
            <style>
              .tbl {{width:100%;border-collapse:collapse;font-family:sans-serif;}}
              .tbl th {{background:#1a1a2e;color:white;padding:10px;text-align:center;}}
              .tbl td {{padding:10px;text-align:center;border-bottom:1px solid #ddd;}}
              .tbl tr:hover {{background:#f5f5f5;}}
              .winner {{background:#ffffff;color:#155724;font-weight:bold;}}
            </style>
            <table class="tbl">
              <tr><th>Criterion - المعيار</th><th>Helsinki</th><th>mBART</th><th>Best - الافضل</th></tr>
              <tr><td>Transaltion - الترجمة</td><td>{res1}</td><td>{res2}</td><td>—</td></tr>
              <tr><td>Speed - السرعة</td><td>{t1} Sec</td><td>{t2} Sec</td><td class="winner">{faster}</td></tr>
              <tr><td>Fluency - الطلاقة</td><td>Machine-like - آلي</td><td>Human-like بشري</td><td class="winner">mBART</td></tr>
              <tr><td>Type - النوع</td><td>Literal - حرفي</td><td>Contextual - سياقي</td><td class="winner">mBART</td></tr>
              <tr><td>Languages - اللغات</td><td>Limited - محدوده</td><td>Multiple - متعدده</td><td class="winner">mBART</td></tr>
            </table>
            """))
        except Exception as e:
            print(f"❌ Error - يوجد خطاء: {e}")

trans_btn.on_click(on_trans)

tab2 = widgets.VBox([
    widgets.HTML("<h3 style='text-align:center'>🌐 Translation - ترجمه 🌐</h3>"),
    widgets.Label("text - النص:"), trans_input,
    trans_btn, trans_out
])

# ── التجميع النهائي ───────────────────────────────────
tabs = widgets.Tab(children=[tab1, tab2])
tabs.set_title(0, "Q & A - سؤال وجواب")
tabs.set_title(1, "Transltion - ترجمة")

display(widgets.VBox([
    widgets.HTML("""
    <div style='text-align:center; padding:15px; background:#1a1a2e; border-radius:10px; margin-bottom:10px;'>
      <h2 style='color:white; margin:0'>🤖 Smart NLP System🤖</h2>
      <p style='color:#aaa; margin:5px 0'>نظام ذكي للسؤال والجواب والترجمة</p>
    </div>
    """),
    tabs,
    widgets.HTML("""
    <div style='text-align:center; margin-top:15px; color:gray; font-size:13px;'>
      Manar Al-Juaidan<br>
      <b>NLP Project @2026</b>
    </div>
    """)
]))